---
# 03. Analysis — AARRR 퍼널 · 리텐션 분석 및 인사이트
---


가입→이력서→지원 퍼널, 전환 소요시간, 코호트 및 Classic/Range/Rolling 리텐션, 원클릭 지원 기능과 미전환 유저 행동을 분석한다.

## 1. 환경 설정 & 분석 데이터 준비

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import koreanize_matplotlib

# notebooks/에서 실행해도 프로젝트 루트의 src 모듈을 불러올 수 있게 설정
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import get_engine, load_logs

warnings.filterwarnings("ignore")

from src.pipeline import build_funnel_data, categorize_url

engine = get_engine()
df = load_logs(engine)

prep = build_funnel_data(df)
df_filtered = prep["df_filtered"]
acquisition = prep["acquisition"]
resume_step1 = prep["resume_step1"]
resume_step2 = prep["resume_step2"]
apply_steps = prep["apply_steps"]
apply_complete = prep["apply_complete"]
funnel = prep["funnel"]


## 2. Acquisition 분석

## 월별 전체 가입자 수

In [ ]:
# 월별 가입자 수
signup_done = df[df['URL'].str.contains('signup/step3/done', case=False, na=False)].copy()

# 유저별 최초 가입 완료 시점만 추출
first_done = (
    signup_done.groupby('user_uuid')['timestamp']
    .min()
    .reset_index()
)
first_done.columns = ['user_uuid', 'first_done']

# 월별 집계
monthly = (
    first_done
    .groupby(first_done['first_done'].dt.to_period('M'))
    .agg(unique_users=('user_uuid', 'nunique'))
    .reset_index()
)
monthly.columns = ['month', 'unique_users']

print(f"전체 합계: {monthly['unique_users'].sum():,}")
print(monthly)

전체 합계: 4,746
      month  unique_users
0   2022-01           317
1   2022-02           237
2   2022-03           270
3   2022-04           250
4   2022-05           245
5   2022-06           243
6   2022-07           217
7   2022-08           227
8   2022-09           169
9   2022-10           172
10  2022-11           189
11  2022-12           226
12  2023-01           220
13  2023-02           249
14  2023-03           218
15  2023-04           197
16  2023-05           178
17  2023-06           179
18  2023-07           130
19  2023-08           166
20  2023-09           156
21  2023-10           142
22  2023-11           118
23  2023-12            31


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(monthly['month'].astype(str), monthly['unique_users'])
ax.plot(monthly['month'].astype(str), monthly['unique_users'], marker='o', linewidth=2)

for i, v in enumerate(monthly['unique_users']):
    ax.text(i, v + 10, str(v), ha='center', fontsize=9)

ax.set_title('Monthly Acquisition')
ax.set_xlabel('Month')
ax.set_ylabel('Unique Users')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 가입 스텝 퍼널 (step1 → step3)

In [ ]:
# signup signup_funnel (step1 -> step3)
# 각 단계별 고유 유저 수
step1 = df[df['URL'].str.contains('signup/step1', case=False, na=False)]['user_uuid'].nunique()
step2 = df[df['URL'].str.contains('signup/step2', case=False, na=False)]['user_uuid'].nunique()
step3 = df[df['URL'].str.contains('signup/step3/done', case=False, na=False)]['user_uuid'].nunique()

signup_funnel = pd.DataFrame({
    'stage': ['signup/step1', 'signup/step2', 'signup/step3/done'],
    'unique_users': [step1, step2, step3]
})

# 전환율 추가
signup_funnel['conversion_rate'] = (signup_funnel['unique_users'] / signup_funnel['unique_users'].iloc[0] * 100).round(1)
signup_funnel['drop_rate'] = (100 - signup_funnel['conversion_rate']).round(1)

print(signup_funnel)

In [ ]:
# signup signup_funnel (step1 -> step3) 시각화
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(signup_funnel['stage'][::-1], signup_funnel['unique_users'][::-1])

for i, (v, r) in enumerate(zip(signup_funnel['unique_users'][::-1], signup_funnel['conversion_rate'][::-1])):
    ax.text(v + 50, i, f'{v:,}명 ({r}%)', va='center', fontsize=10)

ax.set_title('Signup Funnel')
ax.set_xlabel('Unique Users')
ax.set_xlim(0, signup_funnel['unique_users'].max() * 1.2)
plt.tight_layout()
plt.show()

## 3. Activation 분석 — 이력서·지원 퍼널 전환율

In [ ]:
acq_cnt      = len(funnel)
resume1_cnt  = funnel['did_resume1'].sum()
resume2_cnt  = funnel[funnel['did_resume1'] & funnel['did_resume2']].shape[0]
apply1_cnt   = funnel['did_apply1'].sum()
apply2_cnt   = funnel[funnel['did_apply1'] & funnel['did_apply2']].shape[0]
apply3_cnt   = funnel[funnel['did_apply1'] & funnel['did_apply2'] & funnel['did_apply3']].shape[0]
apply4_cnt   = funnel[funnel['did_apply1'] & funnel['did_apply2'] & funnel['did_apply3'] & funnel['did_apply4']].shape[0]
complete_cnt = funnel[funnel['did_apply1'] & funnel['did_complete']].shape[0]

print('=' * 55)
print(f"{'Acquisition (가입완료)':<35} {acq_cnt:>6,}명  100.0%")
print('-' * 55)
print(f"{'이력서 step1':<35} {resume1_cnt:>6,}명  {resume1_cnt/acq_cnt*100:.1f}%")
print(f"{'이력서 step2':<35} {resume2_cnt:>6,}명  {resume2_cnt/acq_cnt*100:.1f}%")
print('-' * 55)
print(f"{'지원 step1':<35} {apply1_cnt:>6,}명  {apply1_cnt/acq_cnt*100:.1f}%")
print(f"{'지원 step2':<35} {apply2_cnt:>6,}명  {apply2_cnt/acq_cnt*100:.1f}%")
print(f"{'지원 step3':<35} {apply3_cnt:>6,}명  {apply3_cnt/acq_cnt*100:.1f}%")
print(f"{'지원 step4':<35} {apply4_cnt:>6,}명  {apply4_cnt/acq_cnt*100:.1f}%")
print(f"{'Activation (지원완료)':<35} {complete_cnt:>6,}명  {complete_cnt/acq_cnt*100:.1f}%")
print('=' * 55)


In [ ]:
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

stages = ['Acquisition\n(가입완료)', '이력서\nstep1', '이력서\nstep2',
          '지원\nstep1', '지원\nstep2', '지원\nstep3', '지원\nstep4', 'Activation\n(지원완료)']
counts = [acq_cnt, resume1_cnt, resume2_cnt, apply1_cnt, apply2_cnt, apply3_cnt, apply4_cnt, complete_cnt]
colors = ['#1F4E79', '#2E75B6', '#2E75B6', '#4BACC6', '#4BACC6', '#4BACC6', '#4BACC6', '#9DC3E6']

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(stages, counts, color=colors, width=0.6, edgecolor='white')

for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{cnt:,}명\n({cnt/acq_cnt*100:.1f}%)',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1F4E79')

ax.set_title('Acquisition → Activation 퍼널', fontsize=14, fontweight='bold', color='#1F4E79')
ax.set_ylabel('유저 수 (명)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, max(counts) * 1.2)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('funnel_final.png', dpi=150, bbox_inches='tight')
plt.show()


### 전환까지 걸린 시간 분포

In [ ]:
activated = funnel[funnel['did_apply1'] & funnel['did_complete']].copy()
activated['signup_to_resume_day'] = (activated['resume_step1_time']   - activated['signup_time']).dt.total_seconds() / 86400
activated['resume_to_apply_day']  = (activated['apply_complete_time'] - activated['resume_step1_time']).dt.total_seconds() / 86400
activated['signup_to_apply_day']  = (activated['apply_complete_time'] - activated['signup_time']).dt.total_seconds() / 86400

activated[['signup_to_resume_day', 'resume_to_apply_day', 'signup_to_apply_day']].describe()


In [ ]:
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

activated['signup_to_resume_day'] = (activated['resume_step1_time'] - activated['signup_time']).dt.total_seconds() / 86400
activated['resume_to_apply_day']  = (activated['apply_complete_time']  - activated['resume_step1_time']).dt.total_seconds() / 86400
activated['signup_to_apply_day']  = (activated['apply_complete_time']  - activated['signup_time']).dt.total_seconds() / 86400


bins     = [0, 1, 3, 7, 30, float('inf')]
labels   = ['1일 이내', '1~3일', '3~7일', '7~30일', '30일 이상']
colors   = ['#1F4E79', '#2E75B6', '#4BACC6', '#9DC3E6', '#D6E4F0']

cols = [
    ('signup_to_resume_day', '가입 → 이력서 작성'),
    ('resume_to_apply_day',  '이력서 → 지원 완료'),
    ('signup_to_apply_day',  '가입 → 지원 완료'),
]

for ax, (col, title) in zip(axes, cols):
    counts = pd.cut(activated[col], bins=bins, labels=labels, right=False).value_counts()[labels]

    bars = ax.bar(labels, counts.values, color=colors, edgecolor='white')

    for bar, cnt in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{cnt:,}명\n({cnt/len(activated)*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1F4E79')

    ax.set_title(title, fontsize=12, fontweight='bold', pad=12, color='#1F4E79')
    ax.set_ylabel('유저 수 (명)', fontsize=10)
    ax.set_xlabel('전환 기간', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, max(counts.values) * 1.25)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('전환 기간 분포', fontsize=14, fontweight='bold', color='#1F4E79', y=1.02)
plt.tight_layout()
plt.savefig('conversion_days_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import platform
import numpy as np

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

bins   = [0, 1, 3, 7, 30, float('inf')]
labels = ['1일 이내', '1~3일', '3~7일', '7~30일', '30일 이상']

activated['구간_가입→이력서'] = pd.cut(activated['signup_to_resume_day'], bins=bins, labels=labels, right=False)
activated['구간_이력서→지원'] = pd.cut(activated['resume_to_apply_day'],  bins=bins, labels=labels, right=False)

heatmap_data = (
    activated
    .groupby(['구간_가입→이력서', '구간_이력서→지원'], observed=True)
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(heatmap_data.values, cmap='Blues', aspect='auto')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('이력서→지원 기간', fontsize=11)
ax.set_ylabel('가입→이력서 기간', fontsize=11)
ax.set_title('전환 기간 조합 분포 (가입→이력서→지원)', fontsize=13, fontweight='bold', color='#1F4E79', pad=15)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = heatmap_data.values[i, j]
        pct = val / len(activated) * 100
        color = 'white' if val > heatmap_data.values.max() * 0.5 else '#1F4E79'
        ax.text(j, i, f'{val:,}\n({pct:.1f}%)', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='유저 수')
plt.tight_layout()
plt.savefig('conversion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 코호트 기반 리텐션 히트맵

In [ ]:
acquisition = acquisition.copy()
acquisition['cohort'] = acquisition['signup_time'].dt.to_period('M')


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

JOB_PATTERN = '|'.join([
    'jobs',
    'api/jobs',
    'companies',
    'api/companies',
    'api/search/jobs',
    'api/search/companies',
    'api/apply_progress',
])

activity_job = df[
    df['URL'].str.split('?').str[0].str.contains(JOB_PATTERN, case=False, na=False)
][['user_uuid', 'timestamp']].copy()

activity_job['activity_date'] = activity_job['timestamp'].dt.normalize()
activity_job = activity_job.merge(acquisition[['user_uuid', 'signup_time', 'cohort']], on='user_uuid', how='inner')
activity_job['days'] = (activity_job['activity_date'] - activity_job['signup_time'].dt.normalize()).dt.days
activity_job['month_num'] = activity_job['days'] // 30
activity_job = activity_job[(activity_job['month_num'] >= 0) & (activity_job['month_num'] <= 24)]

cohort_size = acquisition.groupby('cohort')['user_uuid'].nunique()

cohort_data_job = (
    activity_job.groupby(['cohort', 'month_num'])['user_uuid']
    .nunique()
    .reset_index()
)

cohort_pivot_job = cohort_data_job.pivot(index='cohort', columns='month_num', values='user_uuid').fillna(0)
cohort_pivot_job = cohort_pivot_job.reindex(columns=range(0, 25), fill_value=0)

retention_job = cohort_pivot_job.divide(cohort_size, axis=0) * 100
retention_job = retention_job.drop(columns=[0])

fig, ax = plt.subplots(figsize=(28, 9))
im = ax.imshow(retention_job.values, cmap='Blues', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(24))
ax.set_xticklabels([f'{m}개월' for m in range(1, 25)], fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(len(retention_job.index)))
ax.set_yticklabels([str(c) for c in retention_job.index], fontsize=9)
ax.set_xlabel('가입 후 기간', fontsize=11)
ax.set_ylabel('가입 코호트 (월)', fontsize=11)
ax.set_title('채용 활동 기준 리텐션 히트맵', fontsize=14, fontweight='bold', color='#1F4E79', pad=15)

for i in range(retention_job.shape[0]):
    for j in range(retention_job.shape[1]):
        val = retention_job.values[i, j]
        if not np.isnan(val) and val > 0:
            color = 'white' if val > 50 else '#1F4E79'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='리텐션 (%)')
plt.tight_layout()
plt.savefig('retention_job.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Classic / Range / Rolling 리텐션 비교

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 기준 데이터
acquisition_users = acquisition[['user_uuid', 'signup_time']].copy()
acquisition_users['signup_date'] = acquisition_users['signup_time'].dt.normalize()

activity = df_filtered[['user_uuid', 'timestamp']].copy()
activity['activity_date'] = activity['timestamp'].dt.normalize()
activity = activity.merge(acquisition_users[['user_uuid', 'signup_date']], on='user_uuid', how='inner')
activity['days'] = (activity['activity_date'] - activity['signup_date']).dt.days
activity = activity[activity['days'] >= 0]

# 유저별 날짜 중복 제거 (하루 1회 방문)
activity = activity.drop_duplicates(['user_uuid', 'activity_date'])

total = len(acquisition_users)

# ── 1. Classic Retention ──────────────────────────────
classic_days = [1, 7, 14, 30, 60, 90, 180, 365]
classic_counts = []
for d in classic_days:
    cnt = activity[activity['days'] == d]['user_uuid'].nunique()
    classic_counts.append(cnt)

# ── 2. Range Retention ───────────────────────────────
range_bins = [
    ('Week1\n(1~7일)',    1,   7),
    ('Week2\n(8~14일)',   8,  14),
    ('Month1\n(15~30일)', 15,  30),
    ('Month2\n(31~60일)', 31,  60),
    ('Month3\n(61~90일)', 61,  90),
    ('Month6\n(91~180일)',91, 180),
    ('Year1\n(181~365일)',181,365),
]
range_labels = [r[0] for r in range_bins]
range_counts = []
for label, start, end in range_bins:
    cnt = activity[(activity['days'] >= start) & (activity['days'] <= end)]['user_uuid'].nunique()
    range_counts.append(cnt)

# ── 3. Rolling Retention ──────────────────────────────
rolling_days = [1, 7, 14, 30, 60, 90, 180, 365]
last_active = activity.groupby('user_uuid')['days'].max().reset_index()
last_active.columns = ['user_uuid', 'last_day']
rolling_counts = []
for d in rolling_days:
    cnt = last_active[last_active['last_day'] >= d]['user_uuid'].nunique()
    rolling_counts.append(cnt)

# ── 시각화 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Classic
axes[0].plot([f'Day {d}' for d in classic_days], 
             [c/total*100 for c in classic_counts], 
             marker='o', color='#1F4E79', linewidth=2)
for i, (d, c) in enumerate(zip(classic_days, classic_counts)):
    axes[0].annotate(f'{c/total*100:.1f}%', 
                     (f'Day {d}', c/total*100),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8, color='#1F4E79')
axes[0].set_title('Classic Retention', fontsize=13, fontweight='bold', color='#1F4E79')
axes[0].set_ylabel('리텐션 (%)')
axes[0].set_ylim(0, 110)
axes[0].tick_params(axis='x', rotation=45)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', alpha=0.3, linestyle='--')

# Range
bars = axes[1].bar(range_labels, [c/total*100 for c in range_counts],
                   color='#2E75B6', edgecolor='white')
for bar, cnt in zip(bars, range_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{cnt/total*100:.1f}%',
                 ha='center', va='bottom', fontsize=8, color='#1F4E79', fontweight='bold')
axes[1].set_title('Range Retention', fontsize=13, fontweight='bold', color='#1F4E79')
axes[1].set_ylabel('리텐션 (%)')
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis='x', rotation=0)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].grid(axis='y', alpha=0.3, linestyle='--')

# Rolling
axes[2].plot([f'Day {d}+' for d in rolling_days],
             [c/total*100 for c in rolling_counts],
             marker='o', color='#4BACC6', linewidth=2)
for i, (d, c) in enumerate(zip(rolling_days, rolling_counts)):
    axes[2].annotate(f'{c/total*100:.1f}%',
                     (f'Day {d}+', c/total*100),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8, color='#1F4E79')
axes[2].set_title('Rolling Retention', fontsize=13, fontweight='bold', color='#1F4E79')
axes[2].set_ylabel('리텐션 (%)')
axes[2].set_ylim(0, 110)
axes[2].tick_params(axis='x', rotation=45)
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].grid(axis='y', alpha=0.3, linestyle='--')

plt.suptitle('리텐션 분석 (3종)', fontsize=15, fontweight='bold', color='#1F4E79')
plt.tight_layout()
plt.savefig('retention_3types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

acquisition_users = acquisition[['user_uuid', 'signup_time']].copy()
acquisition_users['signup_date'] = acquisition_users['signup_time'].dt.normalize()

activity = df_filtered[['user_uuid', 'timestamp']].copy()
activity['activity_date'] = activity['timestamp'].dt.normalize()
activity = activity.merge(acquisition_users[['user_uuid', 'signup_date']], on='user_uuid', how='inner')
activity['days'] = (activity['activity_date'] - activity['signup_date']).dt.days
activity = activity[activity['days'] >= 0]
activity = activity.drop_duplicates(['user_uuid', 'activity_date'])

total = len(acquisition_users)
days_list = [1, 3, 7, 14, 30, 60, 90, 180, 365]

# Classic
classic_pct = []
for d in days_list:
    cnt = activity[activity['days'] == d]['user_uuid'].nunique()
    classic_pct.append(cnt / total * 100)

# Range (누적)
range_pct = []
for d in days_list:
    cnt = activity[(activity['days'] >= 1) & (activity['days'] <= d)]['user_uuid'].nunique()
    range_pct.append(cnt / total * 100)

# Rolling
last_active = activity.groupby('user_uuid')['days'].max().reset_index()
last_active.columns = ['user_uuid', 'last_day']
rolling_pct = []
for d in days_list:
    cnt = last_active[last_active['last_day'] >= d]['user_uuid'].nunique()
    rolling_pct.append(cnt / total * 100)

# 시각화
fig, ax = plt.subplots(figsize=(14, 7))

labels = [f'Day {d}' for d in days_list]

ax.plot(labels, classic_pct, marker='o', color='#1F4E79', linewidth=2, label='Classic')
ax.plot(labels, range_pct,   marker='s', color='#4BACC6', linewidth=2, label='Range (누적)')
ax.plot(labels, rolling_pct, marker='^', color='#FF8C00', linewidth=2, label='Rolling')

for i, (c, r, ro) in enumerate(zip(classic_pct, range_pct, rolling_pct)):
    ax.annotate(f'{c:.1f}%', (labels[i], c), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7, color='#1F4E79')
    ax.annotate(f'{r:.1f}%', (labels[i], r), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7, color='#4BACC6')
    ax.annotate(f'{ro:.1f}%',(labels[i], ro),textcoords='offset points', xytext=(0, -14),ha='center', fontsize=7, color='#FF8C00')

ax.set_title('리텐션 타입별 비교', fontsize=14, fontweight='bold', color='#1F4E79')
ax.set_ylabel('리텐션 (%)')
ax.set_xlabel('가입 후 경과일')
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('retention_combined.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
days_list = [1, 3, 7, 14, 30, 60, 90, 180, 365]

print('=' * 55)
print('[Classic Retention]')
print(f"{'기준일':<15} {'접속자':>10} {'비율(%)':>10}")
print('-' * 55)
print(f"{'Day 0':<15} {total:>10,} {'100.0':>10}")
for d, pct in zip(days_list, classic_pct):
    cnt = round(pct * total / 100)
    print(f"{'Day '+str(d):<15} {cnt:>10,} {pct:>10.1f}")

print('\n[Range Retention (누적)]')
print(f"{'기준일':<15} {'접속자':>10} {'비율(%)':>10}")
print('-' * 55)
for d, pct in zip(days_list, range_pct):
    cnt = round(pct * total / 100)
    print(f"{'Day '+str(d):<15} {cnt:>10,} {pct:>10.1f}")

print('\n[Rolling Retention]')
print(f"{'기준일':<15} {'접속자':>10} {'비율(%)':>10}")
print('-' * 55)
print(f"{'Day 0+':<15} {total:>10,} {'100.0':>10}")
for d, pct in zip(days_list, rolling_pct):
    cnt = round(pct * total / 100)
    print(f"{'Day '+str(d)+'+':<15} {cnt:>10,} {pct:>10.1f}")
print('=' * 55)

[Classic Retention]
기준일                    접속자      비율(%)
-------------------------------------------------------
Day 0                4,746      100.0
Day 1                1,827       38.5
Day 3                1,203       25.3
Day 7                1,170       24.7
Day 14               1,035       21.8
Day 30                 715       15.1
Day 60                 429        9.0
Day 90                 323        6.8
Day 180                177        3.7
Day 365                 81        1.7

[Range Retention (누적)]
기준일                    접속자      비율(%)
-------------------------------------------------------
Day 1                1,827       38.5
Day 3                2,584       54.4
Day 7                3,266       68.8
Day 14               3,723       78.4
Day 30               4,052       85.4
Day 60               4,246       89.5
Day 90               4,340       91.4
Day 180              4,467       94.1
Day 365              4,534       95.5

[Rolling Retention]
기준일                    접속

## 6. 원클릭 지원 기능 효과

In [ ]:
# 원클릭 지원 URL이 찍힌 유저 수
oneclick_users = df_filtered[
    df_filtered['URL'].str.split('?').str[0] == 'api/jobs/id/template_oneclick'
]['user_uuid'].nunique()
print(f"원클릭 지원 버튼 클릭 유저: {oneclick_users:,}명")

oneclick_set = set(df_filtered[
    df_filtered['URL'].str.split('?').str[0] == 'api/jobs/id/template_oneclick'
]['user_uuid'])

resume_set = set(resume_step1['user_uuid'])
apply_set  = set(apply_complete['user_uuid'])

oneclick_no_resume = oneclick_set - resume_set
oneclick_applied = oneclick_set & apply_set
oneclick_no_resume_applied = oneclick_no_resume & apply_set

print(f"원클릭 유저 중 이력서 없이 지원완료: {len(oneclick_no_resume_applied):,}명")
print(f"원클릭 유저 중 지원완료: {len(oneclick_applied):,}명")


## 7. 미전환 유저의 마지막 행동 분석

가입은 완료했지만 지원까지 이어지지 않은 유저가 마지막으로 어떤 기능을 사용했는지 확인합니다.

In [ ]:
apply_complete_users = set(apply_complete['user_uuid'])
non_converted = acquisition[~acquisition['user_uuid'].isin(apply_complete_users)]['user_uuid']
print(f"미전환 유저 수: {len(non_converted):,}명")

last_action = (
    df_filtered[df_filtered['user_uuid'].isin(non_converted)]
    .groupby('user_uuid')
    .apply(lambda x: x.loc[x['timestamp'].idxmax(), 'URL'])
    .reset_index()
    .rename(columns={0: 'last_url'})
)
last_action['last_url_clean'] = last_action['last_url'].str.split('?').str[0]
last_action['category'] = last_action['last_url_clean'].apply(categorize_url)

category_counts = last_action['category'].value_counts()
category_counts


In [ ]:
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1F4E79', '#2E75B6', '#4BACC6', '#9DC3E6', '#D6E4F0', '#BDD7EE', '#DDEBF7', '#F2F7FB']

bars = ax.bar(category_counts.index, category_counts.values,
              color=colors[:len(category_counts)], edgecolor='white')

for bar, cnt in zip(bars, category_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{cnt:,}명\n({cnt/len(last_action)*100:.1f}%)',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1F4E79')

ax.set_title('미전환 유저 마지막 행동 카테고리', fontsize=14, fontweight='bold', color='#1F4E79')
ax.set_ylabel('유저 수 (명)')
ax.set_xlabel('마지막 행동')
ax.tick_params(axis='x', rotation=30)
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, max(category_counts.values) * 1.2)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('non_converted_category.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. 핵심 퍼널 요약

In [ ]:
summary = pd.DataFrame({
    'stage': ['Acquisition(가입완료)', '이력서 step1', '이력서 step2', '지원 step1', '지원 step4', 'Activation(지원완료)'],
    'unique_users': [acq_cnt, resume1_cnt, resume2_cnt, apply1_cnt, apply4_cnt, complete_cnt],
})
summary['conversion_rate(%)'] = (summary['unique_users'] / acq_cnt * 100).round(1)
summary


## 9. 인사이트 정리

- 가입 퍼널: step1→step2는 94.3%로 괜찮은데 step2→step3(가입완료)에서 79.1%까지 떨어짐. 마지막 단계(개인정보/약관 등)에서 이탈이 제일 큼
- 이력서 작성: 가입완료 유저 중 62.5%만 이력서 step1까지 감. 애초에 이력서를 안 쓰는 사람이 꽤 많음. 근데 일단 시작하면 step2까지는 87.5%로 잘 넘어감
- 지원 퍼널: step1~2(95.1%, 94.1%)까지는 이탈이 적은데 step3(82.8%)→step4(65.1%)에서 확 빠지고 최종 완료는 34.1%. 지원서 후반부/제출 단계 UX를 손볼 필요가 있어 보임
- 리텐션: Day1 38.5%, Day7 24.7%, Day30 15.1%, Day365 1.7%(classic 기준). 초반 이탈이 크고 이후엔 완만해짐. 근데 rolling 기준으로는 Day180+도 45.1%가 최소 한 번은 다시 옴 → 완전히 나간 게 아니라 가끔 들어오는 유저가 꽤 있다는 뜻
- 미전환 유저 마지막 행동은 프로필/설정, 채용공고 탐색이 제일 많음. 공고까지는 보는데 지원은 안 함. 탐색→지원 사이에 뭔가 더 필요해 보임
- 원클릭 지원: 클릭한 사람(3,094명) 중 거의 다(3,077명) 지원 완료함. 이력서 없이 지원한 사람도 899명. 확실히 전환에 도움 되는 기능인 듯